In [1]:
# Ô 0-1: gói + Java cho SPICE + assert 2 GPU
import subprocess, sys, os, time, glob, json, shutil, re, random
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java; java -version")
sh(f"{sys.executable} -m pip install -q 'transformers==5.18.0' 'peft==0.21.1' 'trl==0.29.1' accelerate bitsandbytes pycocoevalcap sacrebleu")
sh(f"{sys.executable} -m pip uninstall -y torchao")
import torch
assert torch.cuda.device_count() == 2, "⛔ cần T4 ×2"

bin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwl

update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java to provide /usr/bin/java (java) in manual mode
openjdk version "1.8.0_504"
OpenJDK Runtime Environment (build 1.8.0_504-8u504-ga-1ubuntu1~24.04.3-b01)
OpenJDK 64-Bit Server VM (build 25.504-b01, mixed mode)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.8 MB/s eta 0:00:00



Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0



In [2]:
# Ô 0-2: đường dẫn, chép mã, hoà S1, hàm chạy nền
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"
def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g: return g[0]
    raise FileNotFoundError(duoi)
SRC = os.path.dirname(tim("raft_mau.py"))
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
CK500 = next(r for r, d, f in os.walk("/kaggle/input") if "grpo-spice-ck500" in r and "adapter_config.json" in f)
shutil.copytree(SRC, RA, dirs_exist_ok=True)
print("SRC", SRC, "\nBUNDLE", BUNDLE, "\nCK500", CK500, flush=True)
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", M], cwd=RA, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
       "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def chay(script, args, log, gpu):
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                            env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}, cwd=RA, start_new_session=True)
def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120); print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h", [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]
def py(*a):
    r = subprocess.run(["python", *a], capture_output=True, text=True, cwd=RA, env=ENV); print(r.stdout[-6000:], r.stderr[-2000:], flush=True); return r.returncode

SRC /kaggle/input/datasets/uyenle0712/raft-314 
BUNDLE /kaggle/input/datasets/uyenle0712/fgrb-p1-bundle/fgrb_p1_bundle 
CK500 /kaggle/input/datasets/uyenle0712/grpo-spice-ck500


[hoà] xong → /kaggle/working/s1_merged · dtype torch.float16



In [3]:
# Ô R-1: ck500 sinh greedy + 8 mẫu (T = 0,7) trên 2.000 câu nhắc mới, hai GPU
logs = [f"{W}/mau_{i}.log" for i in (0, 1)]
ps = [chay("raft_mau.py", ["--bundle", BUNDLE, "--merged", M, "--ckpt", CK500, "--out", f"{W}/raft_mau_{i}.jsonl", "--shard", f"{i}/2"], logs[i], i) for i in (0, 1)]
doi(ps, logs)
print(open(logs[0]).read()[:2500])

04:19 0.07 h [['  warnings.warn('], ['  warnings.warn(']]


04:21 0.11 h [['  warnings.warn('], ['  warnings.warn(']]


04:23 0.14 h [['  warnings.warn('], ['  warnings.warn(']]


04:25 0.17 h [["  25/1000 · 6.8 phút · 16.22 s/bước · 'Click on the Continue button at the bottom of the '"], ["  25/1000 · 6.1 phút · 14.68 s/bước · 'Click on Set goal at the top right corner.'"]]


04:27 0.21 h [["  25/1000 · 6.8 phút · 16.22 s/bước · 'Click on the Continue button at the bottom of the '"], ["  25/1000 · 6.1 phút · 14.68 s/bước · 'Click on Set goal at the top right corner.'"]]


04:29 0.24 h [["  25/1000 · 6.8 phút · 16.22 s/bước · 'Click on the Continue button at the bottom of the '"], ["  25/1000 · 6.1 phút · 14.68 s/bước · 'Click on Set goal at the top right corner.'"]]


... (127 outputs left out)

08:45 4.51 h [["  975/1000 · 264.3 phút · 16.27 s/bước · 'Click on adi at the bottom of the screen.'"], ['[xong] 1000 dòng → /kaggle/working/raft_mau_1.jsonl']]


08:47 4.54 h [["  975/1000 · 264.3 phút · 16.27 s/bước · 'Click on adi at the bottom of the screen.'"], ['[xong] 1000 dòng → /kaggle/working/raft_mau_1.jsonl']]


08:49 4.57 h [['[xong] 1000 dòng → /kaggle/working/raft_mau_0.jsonl'], ['[xong] 1000 dòng → /kaggle/working/raft_mau_1.jsonl']]


[câu nhắc] hàng [1000, 3000) trên 3991 · shard 0/2: 1000 · bỏ {'trung_val': 0, 'trung_episode_val': 0, 'thieu_anh': 0, 'thieu_ocr': 0, 'rong': 9}
[câu nhắc đầu]
Mục tiêu: I want to find a hotel for my mom at Warsaw for the night of 28th of December as there is a layover for he
Đã làm: Click on the "-" icon in front of Adults option to decrease it to 1. → Click on the Done option. → Click on the Search button.
Chữ đọc được trên màn: 1:13 · Dec 28 · Warsaw, Mazovia, Poland · Dec 29 · USD · Filter · Area · Sort · Hotel · Breakfast Included · Warsaw Old Town · 2 Beds · New User Promo Code Pack · Claim · Diamond Tier Trial Deal Up to 25% 0ff · Promo Code · 319 properties found · Platinum Residence ★★★★ · 4.2/5 · Very Good 27 reviews · Near Warsaw Central Station I Centrum · Saved by 209 people · $66 · Total (incl. taxes): $ 71
Viết câu hướng dẫn cho bước tiếp theo.
[nạp] /kaggle/working/s1_merged · dtype torch.float16 · 4-bit False · Tesla T4
[điểm lưu] /kaggle/input/datasets/uyenle0712/grp

In [4]:
# Ô R-2: chọn đích RAFT + CỔNG 15%
rc = py(f"{RA}/raft_chon.py", "--vao", f"{W}/raft_mau_0.jsonl", f"{W}/raft_mau_1.jsonl", "--out", f"{W}/train_raft.jsonl", "--bao", f"{W}/raft_chon.json")
print(open(f"{W}/raft_chon.json").read())
CONG = rc == 0
print("CỔNG", "ĐẠT" if CONG else "KHÔNG ĐẠT — bỏ qua train", flush=True)

M / 384.5M ( 99.7%)
Progress: 383.5M / 384.5M ( 99.7%)
Progress: 383.5M / 384.5M ( 99.7%)
Progress: 383.5M / 384.5M ( 99.8%)
Progress: 383.5M / 384.5M ( 99.8%)
Progress: 383.5M / 384.5M ( 99.8%)
Progress: 383.5M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.6M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)
Progress: 383.7M / 384.5M ( 99.8%)


{
 "n_cau_nhac": 2000,
 "n_doi": 415,
 "ty_le_doi": 0.2075,
 "dspice_tb_khi_doi": 0.5687,
 "bo_rong": 0,
 "n_train": 2000,
 "G": 8,
 "nguong": 0.15,
 "dat": true
}
CỔNG ĐẠT


In [5]:
# Ô R-3: RAFT — chạy thử 2 bước rồi SFT thật (tiếp adapter ck500, LR 1e-5), tự dừng trước 11 h
if CONG:
    p = chay("ra_train.py", ["--merged", M, "--ckpt", CK500, "--data", f"{W}/train_raft.jsonl", "--lr", "1e-5", "--out", f"{W}/thu_raft", "--n", "32"], f"{W}/thu_raft.log", 0)
    doi([p], [f"{W}/thu_raft.log"]); print(open(f"{W}/thu_raft.log").read()[-1500:], flush=True)
    shutil.rmtree(f"{W}/thu_raft", ignore_errors=True)
    con = 11.0 - (time.time() - T0) / 3600 - 0.4
    p = chay("ra_train.py", ["--merged", M, "--ckpt", CK500, "--data", f"{W}/train_raft.jsonl", "--lr", "1e-5", "--out", f"{W}/raft_sft", "--max-hours", f"{con:.2f}"], f"{W}/train_raft.log", 0)
    doi([p], [f"{W}/train_raft.log"]); print(open(f"{W}/train_raft.log").read()[-2500:], flush=True)
    print("final có:", os.path.exists(f"{W}/raft_sft/final/adapter_model.safetensors"))
for d in (M, RA):
    shutil.rmtree(d, ignore_errors=True)
print(sorted(os.listdir(W)), f"{(time.time()-T0)/3600:.2f} h")

08:55 4.68 h [['  bước 1/2 · loss 0.3134 · |g| 2.184 · lr 2.00e-06 · 103.4 s/bước · VRAM 6.3 GiB']]


08:57 4.71 h [['[xong] 2 bước · 3.6 phút → /kaggle/working/thu_raft/final']]


[dữ liệu] /kaggle/working/train_raft.jsonl · 32 hàng · 2 bước tối ưu × lô 16 · lr 1e-05
[nạp] /kaggle/working/s1_merged · dtype torch.float16 · 4-bit True · Tesla T4
[adapter] /kaggle/input/datasets/uyenle0712/grpo-spice-ck500 · tham số học 14,966,784 (ck500 14,966,784) · dtype tính torch.float16
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[kiểm nhãn] mẫu đầu: 1544 token, tính loss 7 token: 'Click on the OK button.<|im_end|>' · đích 'Click on the OK button.'
  bước 1/2 · loss 0.3134 · |g| 2.184 · lr 2.00e-06 · 103.4 s/bước · VRAM 6.3 GiB
  bước 2/2 · loss 0.1858 · |g| 1.637 · lr 3.00e-06 · 107.2 s/bước · VRAM 6.4 GiB
[xong] 2 bước · 3.6 phút → /kaggle/working/thu_raft/final



08:59 4.74 h [["[kiểm nhãn] mẫu đầu: 1544 token, tính loss 7 token: 'Click on the OK button.<|im_end|>' · đích 'Click on the OK button.'"]]


09:01 4.78 h [['  bước 1/125 · loss 0.3134 · |g| 2.184 · lr 2.00e-06 · 112.4 s/bước · VRAM 6.3 GiB']]


09:03 4.81 h [['  bước 1/125 · loss 0.3134 · |g| 2.184 · lr 2.00e-06 · 112.4 s/bước · VRAM 6.3 GiB']]


... (112 outputs left out)

12:49 8.58 h [['[xong] 125 bước · 230.8 phút → /kaggle/working/raft_sft/final']]


ll be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[kiểm nhãn] mẫu đầu: 1544 token, tính loss 7 token: 'Click on the OK button.<|im_end|>' · đích 'Click on the OK button.'
  bước 1/125 · loss 0.3134 · |g| 2.184 · lr 2.00e-06 · 112.4 s/bước · VRAM 6.3 GiB
  bước 5/125 · loss 0.1991 · |g| 0.848 · lr 6.00e-06 · 111.6 s/bước · VRAM 6.4 GiB
  bước 10/125 · loss 0.2078 · |g| 1.259 · lr 1.00e-05 · 111.3 s/bước · VRAM 6.6 GiB
  bước 15/125 · loss 0.1819 · |g| 1.083 · lr 9.95e-06 · 111.3 s/bước · VRAM 6.6 GiB
  bước 20/125 · loss 0.1705 · |g| 1.354 · lr 9.81e-06 · 111.3 s/bước · VRAM 6.6 GiB
  bước 25/125 · loss 0.1993 · |g| 1.087 · lr 9.59e-06 · 111.2 s/bước · VRAM 6.6 GiB
  bước 30/125 · loss 0.2008 · |g| 0.767 · lr 9.27e-06 · 111.2 s/bước · VRAM 6.6 GiB
  bước 35/125 · loss 0.2150 · |g| 1.985 · lr 8.88e-06 · 111.2 s/bước · VRAM 6.6 GiB
  bước 40/125 · loss 0.1865 · |g| 1.544 · lr 8.41e-06 · 111.0 s/bước · VRAM 6.6 

final có: True


['__notebook__.ipynb', 'mau_0.log', 'mau_1.log', 'raft_chon.json', 'raft_mau_0.jsonl', 'raft_mau_1.jsonl', 'raft_sft', 'thu_raft.log', 'train_raft.jsonl', 'train_raft.log'] 8.58 h


In [ ]:
# Ô R-4: nén kết quả
shutil.make_archive("/tmp/ket_qua_314_R", "zip", W, ".")
shutil.move("/tmp/ket_qua_314_R.zip", f"{W}/ket_qua_314_R.zip")
print(sorted(os.listdir(W)))